# BigAlpha2026 H06 submission

Official-template H06 minute depth-delta OFI proxy. The backend only needs the self-contained `main(datasources, start_date, end_date)` below.

In [ ]:
def main(datasources, start_date, end_date):
    if not isinstance(datasources, dict):
        raise ValueError("datasources must be a dictionary")
    minute_table = datasources.get("bar1m")
    if not isinstance(minute_table, str) or not minute_table.strip():
        raise ValueError("datasources['bar1m'] is required")

    import numpy as np
    import pandas as pd

    OUTPUT_COLUMNS = ["date", "instrument", "factor"]
    UNIVERSE_TABLE = "bigalpha_2026_instruments"

    def _date_bounds(start_date, end_date):
        start = pd.Timestamp(start_date)
        end = pd.Timestamp(end_date)
        if pd.isna(start) or pd.isna(end) or start > end:
            raise ValueError("invalid date range")
        if end == end.normalize():
            end = end + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)
        return start, end

    def _month_chunks(start, end):
        cursor = start
        while not cursor > end:
            chunk_end = min(end, cursor.to_period("M").end_time)
            yield cursor, chunk_end
            cursor = chunk_end + pd.Timedelta(nanoseconds=1)

    def _query(table, sql, start_date, end_date):
        import dai

        return dai.query(
            sql,
            filters={"date": [start_date, end_date]},
            compression=True,
        ).df()

    def _compute_chunk(minute_table, start, end):
        minute_start = start.strftime("%Y-%m-%d %H:%M:%S")
        minute_end = end.strftime("%Y-%m-%d %H:%M:%S.%f")
        day_start = start.date().isoformat()
        day_end = end.date().isoformat()
        minute = _query(
            minute_table,
            f"SELECT date, instrument, bid_volume1, ask_volume1 FROM {minute_table} "
            f"WHERE date >= '{minute_start}' AND NOT date > '{minute_end}' "
            "ORDER BY date, instrument",
            minute_start,
            minute_end,
        )
        universe = _query(
            UNIVERSE_TABLE,
            f"SELECT date, instrument FROM {UNIVERSE_TABLE} "
            f"WHERE date >= '{day_start}' AND NOT date > '{day_end}' "
            "ORDER BY date, instrument",
            day_start,
            day_end,
        )
        required = {"date", "instrument", "bid_volume1", "ask_volume1"}
        if not required.issubset(minute.columns):
            raise ValueError("official minute query lacks required columns")
        if not {"date", "instrument"}.issubset(universe.columns):
            raise ValueError("official universe query lacks required columns")
        work = minute.loc[:, ["date", "instrument", "bid_volume1", "ask_volume1"]].copy()
        work["timestamp"] = pd.to_datetime(work["date"], errors="coerce")
        membership = universe.loc[:, ["date", "instrument"]].copy()
        membership["date"] = pd.to_datetime(membership["date"], errors="coerce").dt.normalize()
        if work["timestamp"].isna().any() or membership["date"].isna().any():
            raise ValueError("official query contains invalid dates")
        if work["instrument"].isna().any() or membership["instrument"].isna().any():
            raise ValueError("official query contains missing instruments")
        work["instrument"] = work["instrument"].astype(str)
        membership["instrument"] = membership["instrument"].astype(str)
        if work.duplicated(["timestamp", "instrument"]).any():
            raise ValueError("official minute query contains duplicate keys")
        if membership.duplicated(["date", "instrument"]).any():
            raise ValueError("official universe contains duplicate membership keys")
        work["date"] = work["timestamp"].dt.normalize()
        work = work.merge(membership, on=["date", "instrument"], how="inner", validate="many_to_one")
        for column in ("bid_volume1", "ask_volume1"):
            work[column] = pd.to_numeric(work[column], errors="coerce")
        bid = work["bid_volume1"]
        ask = work["ask_volume1"]
        valid = np.isfinite(bid) & np.isfinite(ask) & (bid >= 0) & (ask >= 0)
        work["valid_depth"] = valid
        work["depth_pressure"] = (bid - ask).where(valid)
        work = work.sort_values(["date", "instrument", "timestamp"], kind="mergesort").reset_index(drop=True)
        keys = ["date", "instrument"]
        groups = work.groupby(keys, sort=True, group_keys=False)
        previous_valid = groups["valid_depth"].shift(fill_value=False).astype(bool)
        previous_pressure = groups["depth_pressure"].shift()
        work["delta_ofi"] = (work["depth_pressure"] - previous_pressure).where(work["valid_depth"] & previous_valid)
        daily = (
            work.dropna(subset=["delta_ofi"])
            .groupby(keys, sort=True)["delta_ofi"]
            .agg(valid_transitions="count", factor="sum")
            .reset_index()
        )
        daily = daily.loc[(daily["valid_transitions"] >= 1) & np.isfinite(daily["factor"]), OUTPUT_COLUMNS].copy()
        if daily.duplicated(["date", "instrument"]).any():
            raise ValueError("factor output has duplicate keys")
        return daily.sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)

    start, end = _date_bounds(start_date, end_date)
    pieces = []
    for chunk_start, chunk_end in _month_chunks(start, end):
        chunk = _compute_chunk(minute_table, chunk_start, chunk_end)
        if not chunk.empty:
            pieces.append(chunk)
    output = pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame(columns=OUTPUT_COLUMNS)
    if output.duplicated(["date", "instrument"]).any():
        raise ValueError("factor output has duplicate keys across calendar-month chunks")
    return output.loc[:, OUTPUT_COLUMNS].sort_values(["date", "instrument"], kind="mergesort").reset_index(drop=True)
